 ## Initial EDA of cleaned series data


In [57]:
import sys
from pathlib import Path
import datetime as dt
import pandas as pd
from tours import config
import numpy as np
from statsmodels.tsa.seasonal import seasonal_decompose


In [58]:
from tours import dataset
import plotly.express as px


In [59]:
df = dataset.load_clean_data()
df = dataset.cutoff_series(df)
df.head()

,tour_year,tour_month,tour_day,day_name,day_of_week,num_guides,headcount,covid_flag,xmas_flag,source
tour_date,,,,,,,,,,
2018-01-01,2018,1,1,Monday,1,1,37.0,False,False,SS dump
2018-01-02,2018,1,2,Tuesday,2,6,231.0,False,False,SS dump
2018-01-03,2018,1,3,Wednesday,3,5,173.0,False,False,SS dump
2018-01-04,2018,1,4,Thursday,4,4,106.0,False,False,SS dump
2018-01-05,2018,1,5,Friday,5,3,123.0,False,False,SS dump


In [60]:
df.describe()

,tour_year,tour_month,tour_day,day_of_week,num_guides,headcount
count,3135.000000,3135.000000,3135.000000,3135.000000,3135.000000,3135.000000
mean,2021.806061,6.354067,15.720255,3.000957,1.170016,43.576555
std,2.485316,3.428874,8.801826,1.999920,0.979102,35.452421
min,2018.000000,1.000000,1.000000,0.000000,0.000000,0.000000
25%,2020.000000,3.000000,8.000000,1.000000,1.000000,0.000000
50%,2022.000000,6.000000,16.000000,3.000000,1.000000,44.500000
75%,2024.000000,9.000000,23.000000,5.000000,2.000000,66.750000
max,2026.000000,12.000000,31.000000,6.000000,9.000000,348.500000


In [61]:
df.info()

<class 'pandas.DataFrame'>
DatetimeIndex: 3135 entries, 2018-01-01 to 2026-08-01
Freq: D
Data columns (total 10 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   tour_year    3135 non-null   int64  
 1   tour_month   3135 non-null   int64  
 2   tour_day     3135 non-null   int64  
 3   day_name     3135 non-null   str    
 4   day_of_week  3135 non-null   int64  
 5   num_guides   3135 non-null   int64  
 6   headcount    3135 non-null   float64
 7   covid_flag   3135 non-null   bool   
 8   xmas_flag    3135 non-null   bool   
 9   source       3135 non-null   str    
dtypes: bool(2), float64(1), int64(5), str(2)
memory usage: 226.6 KB


In [85]:
fig = px.line(df["headcount"],title="Number of Attendees Jan 2018 - Aug 2026")
fig.show()

Looks like there is a yearly seasonal pattern, and possibly a weekly one too.
Spikes around Christmas new year, secondary spike around April/May? 
Also weekly seasonality?

In [ ]:
num_rows = df.shape[0]
train = df[df.tour_year < 2025]
val = df[df.tour_year == 2025]
test = df[df.tour_year == 2026]

print(f'full df num rows: {num_rows}\n train num rows: {len(train)}\ntrain perc: {len(train)/num_rows*100}')\n
print(f'full df num rows: {num_rows}\n val num rows: {len(val)}\val perc: {len(val)/num_rows*100}')
print(f'full df num rows: {num_rows}\n test num rows: {len(test)}\n test perc: {len(test)/num_rows*100}')


full df num rows: 3135
 train num rows: 2557
train perc: 81.56299840510367
full df num rows: 3135
 val num rows: 365
al perc: 11.64274322169059
full df num rows: 3135
 test num rows: 213
 test perc: 6.794258373205741


In [104]:
from statsmodels.tsa.seasonal import seasonal_decompose
from plotly.subplots import make_subplots
import plotly.graph_objects as go

result = seasonal_decompose(df["headcount"], model='additive', period=365)

fig = make_subplots(rows=3, cols=1, shared_xaxes=True, 
                     subplot_titles=("Trend", "Seasonal (yearly)", "Residual"))

fig.add_trace(go.Scatter(x=result.trend.index, y=result.trend, name="Trend"), row=1, col=1)
fig.add_trace(go.Scatter(x=result.seasonal.index, y=result.seasonal, name="Seasonality"), row=2, col=1)
fig.add_trace(go.Scatter(x=result.resid.index, y=result.resid, name="Residual"), row=3, col=1)

fig.update_layout(height=700, width=1000, title_text = "Yearly Seasonality")
fig.show()



In [105]:
from statsmodels.tsa.seasonal import MSTL

mstl = MSTL(df["headcount"], periods=[7, 365])
result = mstl.fit()

fig = make_subplots(rows=4, cols=1, shared_xaxes=True,
                     subplot_titles=("Trend", "Seasonal (weekly)", "Seasonal (yearly)", "Residual"))

fig.add_trace(go.Scatter(x=result.trend.index, y=result.trend, name="Trend"), row=1, col=1)
fig.add_trace(go.Scatter(x=result.seasonal.index, y=result.seasonal["seasonal_7"], name="Weekly"), row=2, col=1)
fig.add_trace(go.Scatter(x=result.seasonal.index, y=result.seasonal["seasonal_365"], name="Yearly"), row=3, col=1)
fig.add_trace(go.Scatter(x=result.resid.index, y=result.resid, name="Residual"), row=4, col=1)

fig.update_layout(height=1200, width=1000, title_text="MSTL Decomposition (weekly + yearly)")
fig.show()

In [ ]:
fig_zoom = go.Figure()
fig_zoom = make_subplots(rows=2, cols=1, shared_xaxes=True,
                     subplot_titles=("Trend", "Seasonal (weekly)", "Seasonal (yearly)", "Residual"))
fig_zoom.add_trace(go.Scatter(
    x=result.seasonal.index[:180],
    y=result.seasonal["seasonal_7"].iloc[:180],
    mode="lines",
    name = "1st 180 days"
    
))
fig_zoom.add_trace(go.Scatter(
    x=result.seasonal.index[-180:],
    y=result.seasonal["seasonal_7"].iloc[-180:],
    mode="lines"
))
fig_zoom.update_layout(title="Weekly seasonality (zoomed, first 180 days)")
fig_zoom.show()

In [82]:
import pandas as pd
import plotly.express as px

year_mth_avg = df.groupby(["tour_year", "tour_month"])["headcount"].mean().reset_index()

# Make sure month order is correct (1-12), not alphabetical
year_mth_avg = year_mth_avg.sort_values(["tour_year", "tour_month"])

fig = px.line(
    year_mth_avg,
    x="tour_month",
    y="headcount",
    color="tour_year",  # <-- one line per year
    markers=True
)

fig.update_xaxes(
    tickmode="array",
    tickvals=list(range(1, 13)),
    ticktext=["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]
)

fig.update_layout(
    title="Seasonal pattern by year",
    xaxis_title="Month",
    yaxis_title="Average headcount",
    legend_title="Year"
)

fig.show()

In [83]:
from statsmodels.tsa.seasonal import MSTL
import plotly.express as px

df["headcount"] = df["headcount"].fillna(0)  # or .interpolate(), depending on what missing means

mstl = MSTL(df["headcount"], periods=[7, 365])
result = mstl.fit()

# Weekly pattern
fig1 = px.line(result.seasonal["seasonal_7"].iloc[:60], title="Weekly seasonality (first 60 days)")
fig1.show()

# Yearly pattern
fig2 = px.line(result.seasonal["seasonal_365"], title="Yearly seasonality")
fig2.show()